# PART 3 — Transformations (Core)
### PySpark on Databricks — Complete Learning Series

**How to use this notebook:**
1. `Workspace -> Import -> File` → upload this `.py` file (auto-detected as a notebook).
2. Attach to any cluster (fully works on **Databricks Community Edition**).
3. Run top-to-bottom with `Shift+Enter`. This notebook is **self-contained** — Section 0
   builds a rich practice dataset in code, so nothing needs uploading.
4. Practice Questions + Solutions at the end.

**Topics covered in Part 3:**
17. select, selectExpr
18. filter/where (single & multiple conditions)
19. withColumn, withColumnRenamed
20. drop, dropDuplicates, distinct
21. orderBy/sort (asc, desc)
22. limit, sample
23. union, unionByName
24. when/otherwise (conditional columns)
25. cast (type conversion)
26. String functions (concat, substring, trim, upper/lower, split, regexp_replace, regexp_extract)
27. Date/Time functions (to_date, date_add, datediff, year/month/day, current_date)
28. Null handling (na.drop, na.fill, isNull, isNotNull, coalesce)
29. 🎯 Practice Questions with Solutions

## 0. Setup — Build a Rich Practice Dataset
We deliberately include: messy spacing (for `trim`), mixed case text (for `upper`/`lower`),
raw phone strings (for `regexp_extract`/`regexp_replace`), string dates (for date functions),
duplicate rows (for `dropDuplicates`), and NULLs (for null-handling functions) — so every
topic below has real data to work with.

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col

emp_data = [
    (1, "  Amit Sharma",  "IT",     55000, 28, "Pune",      "2021-03-15", "amit.sharma@company.com",   "+91-98765-43210"),
    (2, "Sneha Verma  ",  "HR",     48000, 32, "Mumbai",    "2019-07-01", "SNEHA.VERMA@COMPANY.COM",   "098-7654-3211"),
    (3, "Raj Patel",      "Sales",  62000, 25, "Delhi",     "2022-01-10", "raj.patel@company.com",     "+91 98765 43212"),
    (4, "Priya Singh",    "IT",     71000, 30, "Pune",      "2018-11-23", "priya.singh@company.com",   "9876543213"),
    (5, "Karan Mehta",    "Sales",  None,  27, "Delhi",     "2023-05-05", "karan.mehta@company.com",   "+91-98765-43214"),
    (6, "Neha Gupta",     "HR",     54000, None, "Mumbai",  "2020-09-18", "neha.gupta@company.com",    None),
    (7, "Vikas Rao",      "IT",     67000, 35, None,        "2017-02-28", "vikas.rao@company.com",     "+91-98765-43216"),
    (8, "Meena Iyer",     "Sales",  45000, 24, "Delhi",     "2023-08-30", "meena.iyer@company.com",    "+91-98765-43217"),
    (9, "Raj Patel",      "Sales",  62000, 25, "Delhi",     "2022-01-10", "raj.patel@company.com",     "+91 98765 43212"),  # exact duplicate of row 3
]
emp_columns = ["id", "full_name", "department", "salary", "age", "city", "join_date", "email", "phone_raw"]
df = spark.createDataFrame(emp_data, emp_columns)
df.show(truncate=False)
df.printSchema()

## 17. select() and selectExpr()

| Function | Style | Best for |
|---|---|---|
| `select()` | Pass column objects / names / Column expressions | Programmatic, readable, IDE-autocomplete friendly |
| `selectExpr()` | Pass raw SQL-like STRINGS | Quick SQL-style expressions without importing functions |

**Real-life analogy:** `select()` is like pointing at labeled folders in a filing cabinet.
`selectExpr()` is like handing someone a written SQL formula on a sticky note — Spark parses
the text and figures out what you meant.

In [0]:
# ---- select(): basic + computed columns ----
df.select("full_name", "department").show()                     # plain column selection
df.select(col("full_name"), col("salary")).show()                 # using col()
df.select(df.full_name, (df.salary * 12).alias("annual_salary")).show()   # computed column with alias

In [0]:
# ---- selectExpr(): SQL-string style (no need to import col/expr) ----
df.selectExpr("full_name", "department").show()
df.selectExpr("full_name", "salary * 12 AS annual_salary").show()          # arithmetic + rename, all in one string
df.selectExpr("full_name", "salary", "salary > 50000 AS is_high_earner").show()   # boolean expression as a new column
df.selectExpr("*", "UPPER(department) AS dept_upper").show()                # "*" keeps all existing columns + adds a new one

## 18. filter() / where() — Single & Multiple Conditions
`filter()` and `where()` are 100% identical — `where()` exists only because it reads more
like SQL. Use whichever you find more readable; most PySpark developers use `filter()`.

In [0]:
# ---- Single condition ----
df.filter(col("salary") > 50000).show()                 # greater than
df.where(col("department") == "IT").show()                # equal to

# ---- Multiple conditions: combine with & (AND), | (OR), ~ (NOT) - ALWAYS use parentheses around each condition ----
df.filter((col("department") == "IT") & (col("salary") > 60000)).show()      # AND
df.filter((col("department") == "HR") | (col("department") == "Sales")).show() # OR
df.filter(~(col("department") == "IT")).show()                                 # NOT

# ---- Other common condition helpers ----
df.filter(col("department").isin("IT", "HR")).show()               # value is one of a list
df.filter(col("full_name").contains("Raj")).show()                  # substring match anywhere in the string
df.filter(col("city").isNull()).show()                               # missing city
df.filter(col("salary").between(45000, 65000)).show()                # inclusive range
df.filter("salary > 50000 AND department = 'IT'").show()             # raw SQL-string condition (alternative style)

## 19. withColumn() and withColumnRenamed()

In [0]:
# ---- withColumn(): add / overwrite a column ----
df.withColumn("bonus", col("salary") * 0.10).show()                     # add new column
df.withColumn("salary", col("salary").cast("double")).printSchema()      # overwrite existing column's type

# Chaining multiple withColumn() calls to build several derived columns at once
df_enriched = (
    df
    .withColumn("bonus", col("salary") * 0.10)
    .withColumn("total_pay", col("salary") + col("salary") * 0.10)
)
df_enriched.select("full_name", "salary", "bonus", "total_pay").show()

# ---- withColumnRenamed(): rename ONE column permanently ----
df.withColumnRenamed("full_name", "employee_name").show(2)

# Renaming SEVERAL columns: chain withColumnRenamed calls or loop through a dict
rename_map = {"full_name": "employee_name", "salary": "monthly_salary"}
df_renamed = df
for old, new in rename_map.items():
    df_renamed = df_renamed.withColumnRenamed(old, new)
df_renamed.show(2)

## 20. drop(), dropDuplicates(), distinct()

| Function | Purpose |
|---|---|
| `drop(col)` | Removes ENTIRE column(s) |
| `dropDuplicates([cols])` | Removes duplicate ROWS — considers ALL columns if none specified, or only the given subset |
| `distinct()` | Same as `dropDuplicates()` with NO subset — keeps only unique full rows |

**Real-life analogy:** `drop()` tears out an entire column from a spreadsheet. `distinct()`/
`dropDuplicates()` removes photocopy duplicate ROWS, keeping only one copy of each.

In [0]:
# ---- drop(): remove columns ----
df.drop("phone_raw").show(2)
df.drop("phone_raw", "email").show(2)

In [0]:
# ---- distinct(): unique full rows (remember row 9 is an EXACT duplicate of row 3) ----
print("Row count before distinct:", df.count())
print("Row count after distinct:", df.distinct().count())   # duplicate row 9 collapses into row 3

# ---- dropDuplicates(): default behavior = same as distinct() (checks ALL columns) ----
df.dropDuplicates().show()

# ---- dropDuplicates(subset): remove duplicates based on ONLY specific columns ----
# Example: keep only ONE row per department (first occurrence encountered)
df.dropDuplicates(["department"]).select("id", "full_name", "department").show()

# Example: find employees with the same name+department combo (possible duplicate person entries)
df.dropDuplicates(["full_name", "department"]).count()

## 21. orderBy() / sort() — asc, desc
`orderBy()` and `sort()` are identical (two names, same function).

In [0]:
from pyspark.sql.functions import asc, desc

df.orderBy("salary").show()                        # ascending by default
df.orderBy(col("salary").desc()).show()              # descending using .desc() on the column
df.sort(desc("salary")).show()                        # descending using the desc() function
df.orderBy(col("department").asc(), col("salary").desc()).show()   # multi-column sort: dept A-Z, then salary high-to-low

# NULLS handling in sort order (NULLs sort first by default in ascending order)
df.orderBy(col("age").asc_nulls_last()).show()          # push NULL ages to the END instead of the start
df.orderBy(col("salary").desc_nulls_last()).show()        # push NULL salaries to the END

## 22. limit() and sample()

| Function | Purpose |
|---|---|
| `limit(n)` | Returns the FIRST `n` rows deterministically (like SQL `LIMIT`) |
| `sample(fraction)` | Returns a RANDOM subset of rows — useful for testing on a fraction of a huge dataset |

In [0]:
df.limit(3).show()                                    # first 3 rows only

df.sample(fraction=0.5).show()                          # ~50% of rows, chosen RANDOMLY (count varies each run)
df.sample(fraction=0.5, seed=42).show()                  # seed makes the "random" selection REPRODUCIBLE
df.sample(withReplacement=True, fraction=1.5, seed=1).show()  # withReplacement=True allows the SAME row to be picked more than once

## 23. union() and unionByName()

| Function | Matches columns by | Risk |
|---|---|---|
| `union()` | POSITION (column order) | If column ORDER differs between DataFrames, data silently lands in the WRONG columns! |
| `unionByName()` | COLUMN NAME | Safe even if column order differs; can also handle missing columns with `allowMissingColumns=True` |

**Real-life analogy:** `union()` is like stapling two spreadsheets together assuming column 1
matches column 1 — dangerous if someone reordered a sheet. `unionByName()` matches by the
HEADER LABEL instead, like a smart merge that looks at column titles, not position.

In [0]:
new_hires = spark.createDataFrame([
    (10, "Arjun Nair", "IT", 58000, 26, "Chennai", "2024-01-15", "arjun.nair@company.com", "+91-90000-00001"),
    (11, "Divya Rao",  "HR", 50000, 29, "Chennai", "2024-02-20", "divya.rao@company.com",  "+91-90000-00002"),
], emp_columns)   # SAME column order as df -> union() is safe here

df_combined = df.union(new_hires)
print("Combined row count:", df_combined.count())

# ---- Demonstrating the DANGER of union() with mismatched column order ----
reordered_hires = new_hires.select("full_name", "id", "department", "salary", "age", "city", "join_date", "email", "phone_raw")
# df.union(reordered_hires).show()   # DON'T uncomment blindly - "id" and "full_name" values would SWAP into wrong columns!

# ---- unionByName(): matches by column NAME, safe regardless of column order ----
df_safe_combined = df.unionByName(reordered_hires)
df_safe_combined.select("id", "full_name", "department").show()   # correct - matched by name, not position

# ---- unionByName with allowMissingColumns=True: handles DataFrames with different column SETS ----
partial_data = spark.createDataFrame([(12, "Extra Person", "Sales")], ["id", "full_name", "department"])
df_with_missing = df.unionByName(partial_data, allowMissingColumns=True)   # missing columns filled with NULL automatically
df_with_missing.select("id", "full_name", "department", "salary", "city").show()

## 24. when() / otherwise() — Conditional Columns (like CASE WHEN / IF-ELSE)

In [0]:
from pyspark.sql.functions import when

# Simple 2-branch condition
df.withColumn("salary_flag", when(col("salary") > 55000, "High").otherwise("Normal")).select("full_name", "salary", "salary_flag").show()

# Multi-branch condition (chained .when() calls, evaluated top to bottom, first match wins)
df.withColumn(
    "salary_band",
    when(col("salary").isNull(), "Unknown")
    .when(col("salary") >= 60000, "High")
    .when(col("salary") >= 48000, "Medium")
    .otherwise("Low")
).select("full_name", "salary", "salary_band").show()

# when() combined with multiple conditions using & / |
df.withColumn(
    "priority_review",
    when((col("department") == "IT") & (col("salary") > 65000), "Yes").otherwise("No")
).select("full_name", "department", "salary", "priority_review").show()

# when() WITHOUT otherwise() -> unmatched rows become NULL
df.withColumn("is_pune", when(col("city") == "Pune", True)).select("full_name", "city", "is_pune").show()

## 25. cast() — Type Conversion
Two equivalent syntaxes: `col("x").cast("double")` (string type name) or
`col("x").cast(DoubleType())` (imported type object). Both work identically.

In [0]:
from pyspark.sql.types import DoubleType, IntegerType, StringType, DateType

df.printSchema()   # notice: salary is currently "long", age is "long", join_date is currently "string"

# ---- Using string type names ----
df_cast1 = df.withColumn("salary", col("salary").cast("double")) \
             .withColumn("age", col("age").cast("int"))
df_cast1.printSchema()

# ---- Using imported type objects (equivalent) ----
df_cast2 = df.withColumn("salary", col("salary").cast(DoubleType())) \
             .withColumn("id", col("id").cast(StringType()))
df_cast2.printSchema()

# ---- Casting a string to an actual DATE type (needed before using date functions in Section 27) ----
df_cast3 = df.withColumn("join_date", col("join_date").cast(DateType()))
df_cast3.printSchema()
df_cast3.select("full_name", "join_date").show()

# ---- What happens when a cast is IMPOSSIBLE? Returns NULL instead of crashing (silent failure - be careful!) ----
bad_cast_demo = spark.createDataFrame([("abc",)], ["text_value"])
bad_cast_demo.withColumn("as_number", col("text_value").cast("int")).show()   # "abc" -> NULL, no error thrown

## 26. String Functions
concat, substring, trim, upper/lower, split, regexp_replace, regexp_extract — and more.

In [0]:
from pyspark.sql.functions import (
    concat, concat_ws, substring, trim, ltrim, rtrim, upper, lower, initcap,
    split, regexp_replace, regexp_extract, length, lpad, rpad
)

# ---- trim/ltrim/rtrim: remove leading/trailing whitespace (our data has messy spacing on purpose!) ----
df.select(
    col("full_name"),
    trim(col("full_name")).alias("trimmed"),                 # removes BOTH leading and trailing spaces
    length(col("full_name")).alias("original_length"),        # length INCLUDES the extra spaces
    length(trim(col("full_name"))).alias("trimmed_length")     # length AFTER trimming
).show()

In [0]:
# ---- upper/lower/initcap: change case ----
df.select(
    col("email"),
    upper(col("email")).alias("email_upper"),
    lower(col("email")).alias("email_lower"),
    initcap(trim(col("full_name"))).alias("name_title_case")    # capitalizes first letter of EACH word
).show(truncate=False)

In [0]:
# ---- concat / concat_ws: joining strings together ----
df.select(
    concat(trim(col("full_name")), lit(" - "), col("department")).alias("label_v1"),   # concat() needs manual separators via lit()
    concat_ws(" | ", trim(col("full_name")), col("department"), col("city")).alias("label_v2")  # concat_ws() adds a separator AUTOMATICALLY, and skips NULLs gracefully!
).show(truncate=False)

In [0]:
# ---- substring: extract part of a string by position ----
# substring(column, start_position, length)  -- NOTE: position is 1-indexed (not 0-indexed like Python!)
df.select(
    col("email"),
    substring(col("email"), 1, 5).alias("first_5_chars")     # first 5 characters
).show(truncate=False)

# ---- split: break a string into an array using a delimiter ----
df_split = df.select(
    col("email"),
    split(col("email"), "@").alias("email_parts")             # splits "user@domain.com" -> ["user", "domain.com"]
)
df_split.show(truncate=False)
df_split.select(
    col("email"),
    col("email_parts")[0].alias("username"),                    # array index 0 -> part before @
    col("email_parts")[1].alias("domain")                        # array index 1 -> part after @
).show(truncate=False)

In [0]:
# ---- regexp_replace: find-and-replace using a REGEX pattern (great for cleaning messy text) ----
# Our phone_raw column has INCONSISTENT formats: "+91-98765-43210", "098-7654-3211", "9876543213", etc.
# Goal: strip everything except digits, to get a clean 10-13 digit number.
df.select(
    col("phone_raw"),
    regexp_replace(col("phone_raw"), "[^0-9]", "").alias("digits_only")   # [^0-9] means "anything that is NOT a digit" -> replace with ""
).show(truncate=False)

# Another common use: masking sensitive data
df.select(
    col("email"),
    regexp_replace(col("email"), "^[^@]+", "***").alias("masked_email")   # replace everything before "@" with "***"
).show(truncate=False)

In [0]:
# ---- regexp_extract: PULL OUT a piece of a string that matches a regex GROUP ----
# regexp_extract(column, pattern, groupIndex) -- groupIndex 0 = whole match, 1 = first (...) group, etc.
df.select(
    col("phone_raw"),
    regexp_extract(col("phone_raw"), r"(\d{2,3})[-\s]?(\d{4,5})[-\s]?(\d{4,5})", 0).alias("full_match")
).show(truncate=False)

# Extract just the domain name from an email using a capture group
df.select(
    col("email"),
    regexp_extract(col("email"), r"@([A-Za-z0-9.]+)", 1).alias("domain_only")   # group 1 = text captured by (...)
).show(truncate=False)

In [0]:
# ---- lpad / rpad: pad a string to a fixed length (useful for formatting IDs, codes) ----
df.select(
    col("id"),
    lpad(col("id").cast("string"), 5, "0").alias("padded_id")   # "1" -> "00001" (pad LEFT with "0" to reach length 5)
).show()

## 27. Date/Time Functions
to_date, date_add, datediff, year/month/day, current_date — and more.

In [0]:
from pyspark.sql.functions import to_date, date_add, date_sub, datediff, year, month, dayofmonth, current_date, current_timestamp, months_between, date_format

# First, convert the string "join_date" column into a REAL DateType column (required for date math!)
df_dates = df.withColumn("join_date", to_date(col("join_date"), "yyyy-MM-dd"))
df_dates.select("full_name", "join_date").show()
df_dates.printSchema()   # join_date is now "date" type, not "string"

In [0]:
# ---- current_date() / current_timestamp(): today's date / exact current moment ----
df_dates.select(
    col("full_name"),
    current_date().alias("todays_date"),           # same value for every row - today's date, no time component
    current_timestamp().alias("current_moment")      # includes date + time + timezone
).show(truncate=False)

In [0]:
# ---- year() / month() / dayofmonth(): extract date parts ----
df_dates.select(
    col("join_date"),
    year(col("join_date")).alias("join_year"),
    month(col("join_date")).alias("join_month"),
    dayofmonth(col("join_date")).alias("join_day")
).show()

In [0]:
# ---- date_add() / date_sub(): add or subtract N days from a date ----
df_dates.select(
    col("join_date"),
    date_add(col("join_date"), 90).alias("probation_end_90_days"),    # 90 days after join_date
    date_sub(col("join_date"), 30).alias("30_days_before_join")        # 30 days BEFORE join_date
).show()

In [0]:
# ---- datediff(): number of days BETWEEN two dates (end_date, start_date) ----
df_dates.select(
    col("full_name"),
    col("join_date"),
    datediff(current_date(), col("join_date")).alias("days_employed")    # today MINUS join_date, in days
).show()

# ---- months_between(): fractional number of months between two dates - handy for tenure calculations ----
df_dates.select(
    col("full_name"),
    col("join_date"),
    months_between(current_date(), col("join_date")).alias("months_employed")
).show()

In [0]:
# ---- date_format(): convert a date into a CUSTOM STRING format ----
df_dates.select(
    col("join_date"),
    date_format(col("join_date"), "dd/MM/yyyy").alias("indian_format"),      # 2021-03-15 -> 15/03/2021
    date_format(col("join_date"), "MMM dd, yyyy").alias("readable_format"),   # 2021-03-15 -> Mar 15, 2021
    date_format(col("join_date"), "yyyy-MM").alias("year_month")               # 2021-03-15 -> 2021-03 (great for monthly grouping)
).show()

## 28. Null Handling — na.drop(), na.fill(), isNull(), isNotNull(), coalesce()

| Function | Purpose |
|---|---|
| `df.na.drop()` | REMOVES rows containing NULL(s) | 
| `df.na.fill(value)` | REPLACES NULLs with a given value |
| `col.isNull()` / `col.isNotNull()` | Boolean check — USE inside `filter()` |
| `coalesce(colA, colB, ...)` | Returns the FIRST non-null value across several columns |

Remember from Section 0: row 5 has NULL `salary`, row 6 has NULL `age`, row 7 has NULL `city` and `phone_raw`.

In [0]:
df.select("id", "full_name", "salary", "age", "city", "phone_raw").show()

In [0]:
# ---- isNull() / isNotNull(): filtering ----
df.filter(col("salary").isNull()).select("id", "full_name", "salary").show()          # rows missing salary
df.filter(col("city").isNotNull()).select("id", "full_name", "city").show()             # rows WITH a city

# Count NULLs per column - very common data-quality check
from pyspark.sql.functions import count, when as when_fn
null_counts = df.select([count(when_fn(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

In [0]:
# ---- na.drop(): remove rows with NULLs ----
df.na.drop().count()                                    # drops ANY row with AT LEAST ONE null (default "any")
df.na.drop(how="all").count()                             # drops a row ONLY IF ALL its columns are null (rare)
df.na.drop(subset=["salary"]).count()                       # drops rows ONLY if "salary" specifically is null
df.na.drop(subset=["salary", "age", "city"], thresh=2).count()   # keeps a row if AT LEAST 2 of these 3 columns are non-null

In [0]:
# ---- na.fill(): replace NULLs with a default value ----
df.na.fill(0, subset=["salary"]).select("id", "full_name", "salary").show()             # NULL salary -> 0
df.na.fill("Unknown", subset=["city"]).select("id", "full_name", "city").show()           # NULL city -> "Unknown"

# Fill MULTIPLE columns at once with DIFFERENT default values using a dictionary
df.na.fill({"salary": 0, "city": "Unknown", "phone_raw": "N/A"}).select("id", "full_name", "salary", "city", "phone_raw").show()

In [0]:
# ---- coalesce(): pick the first NON-NULL value across multiple columns (great for fallback logic) ----
from pyspark.sql.functions import coalesce, lit

# Example: if "city" is missing, fall back to "Head Office" as a default (without permanently overwriting real data elsewhere)
df.withColumn("city_display", coalesce(col("city"), lit("Head Office"))).select("full_name", "city", "city_display").show()

# Example: build ONE "best available contact" column from phone OR email
df.withColumn("best_contact", coalesce(col("phone_raw"), col("email"))).select("full_name", "phone_raw", "email", "best_contact").show(truncate=False)

# 🎯 PRACTICE QUESTIONS — Part 3
Try each question yourself first, then check the solution cell below it. All questions reuse `df` from Section 0.

| Q# | Question | Expected Result / Columns |
|---|---|---|
| 1 | Using `selectExpr`, create a column `annual_salary` = salary * 12, keep `full_name` too | full_name, annual_salary |
| 2 | Filter employees in "Sales" OR "HR" department AND age > 25 | matching rows |
| 3 | Add a column `email_domain` extracted from the email using `split` | full_name, email, email_domain |
| 4 | Remove exact duplicate rows from `df` and show the resulting row count | should be 8 (down from 9) |
| 5 | Sort employees by department (A-Z), then by salary (highest first) | sorted rows |
| 6 | Get a reproducible random 50% sample of `df` using `seed=100` | ~4-5 rows, same every run |
| 7 | Union `df` with a new 1-row DataFrame safely even if its columns are in a different order | uses unionByName |
| 8 | Create a column `experience_level`: 'Senior' if age>=30, 'Mid' if age>=27, else 'Junior', NULL age -> 'Unknown' | full_name, age, experience_level |
| 9 | Cast `salary` to `IntegerType` and `join_date` (string) to `DateType` | printSchema shows new types |
| 10 | Clean `full_name` by trimming spaces and converting to Title Case | full_name, clean_name |
| 11 | Use `regexp_replace` to strip all non-digit characters from `phone_raw` | full_name, phone_raw, phone_clean |
| 12 | Use `regexp_extract` to pull just the username part of the email (before the @) | full_name, email, username |
| 13 | Add a column `join_year` and a column `tenure_days` (days since join_date until today) | full_name, join_year, tenure_days |
| 14 | Fill NULL `salary` with the value 40000 AND NULL `city` with "Unknown" in one call | full_name, salary, city |
| 15 | Create `contact_info` using `coalesce` on phone_raw then email, defaulting to "No Contact" if both are null | full_name, contact_info |

---
## ✅ SOLUTIONS

In [0]:
# Q1
df.selectExpr("full_name", "salary * 12 AS annual_salary").show()

In [0]:
# Q2
df.filter(((col("department") == "Sales") | (col("department") == "HR")) & (col("age") > 25)).show()

In [0]:
# Q3
df.withColumn("email_parts", split(col("email"), "@")) \
  .withColumn("email_domain", col("email_parts")[1]) \
  .select("full_name", "email", "email_domain").show(truncate=False)

In [0]:
# Q4
df_no_dupes = df.distinct()
print("Row count:", df_no_dupes.count())

In [0]:
# Q5
df.orderBy(col("department").asc(), col("salary").desc()).select("full_name", "department", "salary").show()

In [0]:
# Q6
df.sample(fraction=0.5, seed=100).show()

In [0]:
# Q7
extra_row = spark.createDataFrame([("Extra Employee", 13, "Marketing")], ["full_name", "id", "department"])
df.unionByName(extra_row, allowMissingColumns=True).select("id", "full_name", "department").show()

In [0]:
# Q8
df.withColumn(
    "experience_level",
    when(col("age").isNull(), "Unknown")
    .when(col("age") >= 30, "Senior")
    .when(col("age") >= 27, "Mid")
    .otherwise("Junior")
).select("full_name", "age", "experience_level").show()

In [0]:
# Q9
df_q9 = df.withColumn("salary", col("salary").cast(IntegerType())) \
          .withColumn("join_date", col("join_date").cast(DateType()))
df_q9.printSchema()

In [0]:
# Q10
df.withColumn("clean_name", initcap(trim(col("full_name")))).select("full_name", "clean_name").show()

In [0]:
# Q11
df.withColumn("phone_clean", regexp_replace(col("phone_raw"), "[^0-9]", "")).select("full_name", "phone_raw", "phone_clean").show(truncate=False)

In [0]:
# Q12
df.withColumn("username", regexp_extract(col("email"), r"^([^@]+)@", 1)).select("full_name", "email", "username").show(truncate=False)

In [0]:
# Q13
df.withColumn("join_date_typed", to_date(col("join_date"), "yyyy-MM-dd")) \
  .withColumn("join_year", year(col("join_date_typed"))) \
  .withColumn("tenure_days", datediff(current_date(), col("join_date_typed"))) \
  .select("full_name", "join_year", "tenure_days").show()

In [0]:
# Q14
df.na.fill({"salary": 40000, "city": "Unknown"}).select("full_name", "salary", "city").show()

In [0]:
# Q15
df.withColumn("contact_info", coalesce(col("phone_raw"), col("email"), lit("No Contact"))).select("full_name", "contact_info").show(truncate=False)

## ✅ Part 3 Summary — What You Learned
- **select vs selectExpr** — column-object style vs SQL-string style
- **filter/where** with AND/OR/NOT, `isin`, `contains`, `between`, `isNull`
- **withColumn / withColumnRenamed** — adding, overwriting, chaining, bulk renaming
- **drop vs dropDuplicates vs distinct** — column removal vs row de-duplication (full or subset)
- **orderBy/sort** — asc/desc, multi-column sort, NULL ordering control
- **limit vs sample** — deterministic top-N vs random subsets (with reproducible `seed`)
- **union vs unionByName** — why position-based union is DANGEROUS, and how name-based union protects you
- **when/otherwise** — multi-branch conditional columns, like SQL CASE WHEN
- **cast()** — type conversion, and the silent-NULL behavior on invalid casts
- **String functions**: trim family, upper/lower/initcap, concat/concat_ws, substring, split,
  regexp_replace (cleaning), regexp_extract (pulling out patterns), lpad/rpad
- **Date functions**: to_date, current_date/current_timestamp, year/month/dayofmonth,
  date_add/date_sub, datediff, months_between, date_format
- **Null handling**: na.drop (any/all/subset/thresh), na.fill (single + dictionary), isNull/isNotNull, coalesce fallback pattern
- **15 hands-on practice questions with full solutions**

### 🔜 Coming in Part 4: Aggregations & Grouping
groupBy + agg, multiple aggregations, pivot/unpivot, rollup/cube, and Window functions
(row_number, rank, dense_rank, lag, lead) — with the same style: full examples + practice questions.